# 04 Baseline Poisson Naïve Bayes Decoder (ThresholdCrossings, γ = 0)

This notebook implements the **baseline decoder** described in Issar et al. (2020),
*J Neurophysiol* 123:1472–1485.

## Context
The paper trains a neural network (NAS — Not A Sorter) that assigns each waveform
a spike-likelihood score Pspike ∈ [0,1].  A γ-threshold gates which waveforms are
used for decoding.  **γ = 0 means all threshold crossings are used** — this is the
baseline from which all NAS improvements are measured.

This notebook computes that baseline: a **Poisson Naïve Bayes decoder**
(paper §Materials and Methods, "Decoding") applied to the ThresholdCrossings
stream across all 312 sessions of Monkey N.

## Inputs
- `/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/session_master_index.csv`
- `/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/decoder_trial_table.csv`
- `/kaggle/input/datasets/katakuricharlotte/03-stream-inventory-results/tables_stream_inventory/candidate_stream_manifest.csv`
- Raw NWB files: `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N`

## Outputs
- `tables_baseline_decoder/session_baseline_accuracy.csv`
- `tables_baseline_decoder/decoder_unit_table.csv`
- `meta_baseline_decoder/baseline_decoder_report.json`
- `figures_baseline_decoder/fig01_*.{png,pdf}` … `fig07_*`

## Figures (paper-aligned)
| Figure | Paper analogue |
|---|---|
| fig01: Sample session accuracy report | Fig. 4A/B (γ = 0 baseline point) |
| fig02: Session-wise accuracy × trial count | Fig. 4A/B dual-axis |
| fig03: Accuracy vs days since first session | Fig. 5C |
| fig04: Early vs late session distributions | Fig. 5D |
| fig05: Session distribution histograms (4-panel) | Fig. 4C–F |
| fig06: Mean ± 1 SE normalised accuracy | Fig. 5E (γ = 0 slice) |
| fig07: Accuracy vs recording quality proxy | Fig. 5A/B |

In [45]:
!pip install pynwb h5py

In [46]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

warnings.filterwarnings("ignore")
from pynwb import NWBHDF5IO

In [47]:
# ─────────────────────────────────────────────────────────────────────────────
# PAPER-STYLE GLOBAL SETTINGS — Issar et al. 2020 (J Neurophysiol)
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": 11,
    "legend.frameon": True,
    "legend.edgecolor": "0.4",
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

C_BLACK   = "#1a1a1a"
C_MAROON  = "#8B2222"     # TC / primary accuracy
C_BLUE    = "#2166AC"     # baseline reference dot (paper Fig 4A)
C_GREEN   = "#1B7837"     # maximum / best accuracy
C_GRAY    = "#888888"     # chance level / control
C_ORANGE  = "#E08214"     # SpikingBandPower stream
C_EARLY   = "#2166AC"     # early sessions (blue, as in paper Fig 5E)
C_LATE    = "#8B2222"     # late sessions  (maroon, as in paper Fig 5E)

MARKER_TC  = "o"
MARKER_SBP = "s"
CHANCE     = 0.125        # 1/8 directions

def paper_axes(ax, top=True, right=True):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8, alpha=0.85)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=top, right=right)

np.random.seed(42)

## Paths

In [48]:
DATASET_DIR = Path("/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N")

IN_IDX_DIR    = Path("/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index")
IN_STREAM_DIR = Path("/kaggle/input/datasets/katakuricharlotte/03-stream-inventory-results/tables_stream_inventory")

SESSION_MASTER_CSV    = IN_IDX_DIR    / "session_master_index.csv"
DECODER_TRIAL_CSV     = IN_IDX_DIR    / "decoder_trial_table.csv"
CANDIDATE_STREAM_CSV  = Path("/kaggle/input/datasets/katakuricharlotte/03-neural-streaming-results/tables_stream_inventory/candidate_stream_manifest.csv")

OUT_FIG_DIR   = Path("/kaggle/working/figures_baseline_decoder")
OUT_TABLE_DIR = Path("/kaggle/working/tables_baseline_decoder")
OUT_META_DIR  = Path("/kaggle/working/meta_baseline_decoder")

for d in [OUT_FIG_DIR, OUT_TABLE_DIR, OUT_META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAMPLE_NWB = DATASET_DIR / "sub-Monkey-N_ses-20200127_ecephys.nwb"

print("DATASET_DIR     :", DATASET_DIR)
print("SESSION_MASTER  :", SESSION_MASTER_CSV)
print("DECODER_TRIALS  :", DECODER_TRIAL_CSV)
print("CANDIDATE_STREAM:", CANDIDATE_STREAM_CSV)
print("SAMPLE_NWB      :", SAMPLE_NWB)

DATASET_DIR     : /kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N
SESSION_MASTER  : /kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/session_master_index.csv
DECODER_TRIALS  : /kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/decoder_trial_table.csv
CANDIDATE_STREAM: /kaggle/input/datasets/katakuricharlotte/03-neural-streaming-results/tables_stream_inventory/candidate_stream_manifest.csv
SAMPLE_NWB      : /kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N/sub-Monkey-N_ses-20200127_ecephys.nwb


In [49]:
assert DATASET_DIR.exists(),           "Missing dataset directory"
assert SESSION_MASTER_CSV.exists(),    "Missing session master CSV (run notebook 02)"
assert DECODER_TRIAL_CSV.exists(),     "Missing decoder trial table (run notebook 02)"
assert CANDIDATE_STREAM_CSV.exists(),  "Missing candidate stream manifest (run notebook 03)"
assert SAMPLE_NWB.exists(),            "Missing sample NWB"

session_df  = pd.read_csv(SESSION_MASTER_CSV)
trial_df    = pd.read_csv(DECODER_TRIAL_CSV)
stream_df   = pd.read_csv(CANDIDATE_STREAM_CSV)

session_df["session_date"] = pd.to_datetime(session_df["session_date"], errors="coerce")
if "days_since_first_session" not in session_df.columns:
    d0 = session_df["session_date"].min()
    session_df["days_since_first_session"] = (session_df["session_date"] - d0).dt.days

tc_stream_df = stream_df[stream_df["name"] == "ThresholdCrossings"].copy().reset_index(drop=True)

print("session_df shape  :", session_df.shape)
print("trial_df shape    :", trial_df.shape)
print("stream_df shape   :", stream_df.shape)
print("TC manifest rows  :", len(tc_stream_df))
display(session_df.head(3))
display(trial_df.head(3))

session_df shape  : (312, 22)
trial_df shape    : (117000, 18)
stream_df shape   : (624, 25)
TC manifest rows  : 312


,file_name,file_path,session_date,identifier,session_description,session_start_time,institution,lab,subject_id,subject_species,...,n_electrodes,n_acquisition_streams,n_processing_modules,n_intervals_tables,trials_present,units_present,electrodes_present,target_style_inferred,session_index,days_since_first_session
0,sub-Monkey-N_ses-20200127_ecephys.nwb,/kaggle/input/datasets/katakuricharlotte/dandi...,2020-01-27,2020-01-27_CO_nwb,Neural and behavioral data for target style CO,2020-01-27 00:00:00+00:00,University of Michigan,Chestek Lab,Monkey N,Macaca mulatta,...,96,0,0,1,True,False,True,CO,1,0
1,sub-Monkey-N_ses-20200130_ecephys.nwb,/kaggle/input/datasets/katakuricharlotte/dandi...,2020-01-30,2020-01-30_CO_nwb,Neural and behavioral data for target style CO,2020-01-30 00:00:00+00:00,University of Michigan,Chestek Lab,Monkey N,Macaca mulatta,...,96,0,0,1,True,False,True,CO,2,3
2,sub-Monkey-N_ses-20200204_ecephys.nwb,/kaggle/input/datasets/katakuricharlotte/dandi...,2020-02-04,2020-02-04_CO_nwb,Neural and behavioral data for target style CO,2020-02-04 00:00:00+00:00,University of Michigan,Chestek Lab,Monkey N,Macaca mulatta,...,96,0,0,1,True,False,True,CO,3,8


,file_name,session_index,session_date,identifier,target_style_inferred,id,start_time,stop_time,trial_duration,trial_number,trial_count,run_id,index_target_position,mrs_target_position,target_style,trial_timeout,timeseries_n_items,file_path
0,sub-Monkey-N_ses-20200127_ecephys.nwb,1,2020-01-27,2020-01-27_CO_nwb,CO,0,8.41,9.67,1.26,6.0,64,3,0.3,0.3,CO,30000.0,6,/kaggle/input/datasets/katakuricharlotte/dandi...
1,sub-Monkey-N_ses-20200127_ecephys.nwb,1,2020-01-27,2020-01-27_CO_nwb,CO,1,9.69,11.05,1.36,7.0,69,3,0.5,0.5,CO,30000.0,6,/kaggle/input/datasets/katakuricharlotte/dandi...
2,sub-Monkey-N_ses-20200127_ecephys.nwb,1,2020-01-27,2020-01-27_CO_nwb,CO,2,11.07,12.95,1.88,8.0,95,3,0.7,0.5,CO,30000.0,6,/kaggle/input/datasets/katakuricharlotte/dandi...


## Helper functions

### Neural data extraction

For each session we load the `ThresholdCrossings` `ElectricalSeries` from the NWB file.
The stream shape is **(T × 96)**, where T is the number of samples.  We align samples to
trial timing using the `timestamps` array (if present) or the stream's `rate` and
`starting_time`.

### Decoding window

Following Issar et al. (2020) §Materials and Methods:
> *"Waveforms classified as spikes from 300 ms to 500 ms after fixation
> (i.e., 50 ms after stimulus offset) were used for decoding."*

The memory-guided saccade task timing is:
- Fixation: 200 ms → Stimulus flash: 50 ms → Delay: 500 ms
- Decoding window: [300 ms, 500 ms] relative to fixation onset

For the NWB trial table, `start_time` marks trial start (≈ fixation onset).
We extract TC counts in `[start + 0.30, start + 0.50]` seconds.

### Target label extraction

`mrs_target_position` stores (x, y) target coordinates.  We compute the polar angle
and discretise into **8 octants** (0°, 45°, 90°, …, 315°) for the center-out task.

In [50]:
def safe_attr(obj, attr, default=None):
    try:
        return getattr(obj, attr)
    except Exception:
        return default

def get_tc_stream_meta(nwb_path):
    """Return (rate_hz, starting_time, has_timestamps) for ThresholdCrossings."""
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()
        for obj in nwb.all_children():
            if safe_attr(obj, "name") == "ThresholdCrossings":
                rate  = safe_attr(obj, "rate",          None)
                t0    = safe_attr(obj, "starting_time", 0.0)
                ts    = safe_attr(obj, "timestamps",    None)
                has_ts = ts is not None
                return rate, t0, has_ts
    return None, 0.0, False

def angle_to_octant(angle_deg):
    """Snap angle to nearest of 8 octants: 0,45,90,...,315."""
    octant = int((angle_deg + 22.5) % 360 // 45)
    return octant  # 0..7

def extract_target_labels(trials_df):
    """
    Discretise mrs_target_position (or index_target_position) into 8 direction labels.
    Returns a Series of integer labels 0–7, or NaN if unavailable.
    """
    # Try index_target_position first (already discrete in CO tasks)
    if "index_target_position" in trials_df.columns:
        labels = pd.to_numeric(trials_df["index_target_position"], errors="coerce")
        if labels.notna().mean() > 0.8 and labels.nunique() <= 12:
            return labels.astype("Int64")

    # Try mrs_target_position as (x, y) string or separate columns
    for col in ["mrs_target_position", "target_position"]:
        if col not in trials_df.columns:
            continue
        try:
            pos = trials_df[col].apply(
                lambda v: [float(x) for x in str(v).strip("[]() ").split()] if pd.notna(v) else [np.nan, np.nan]
            )
            x_vals = pos.apply(lambda p: p[0] if len(p) >= 1 else np.nan)
            y_vals = pos.apply(lambda p: p[1] if len(p) >= 2 else np.nan)
            angles = np.degrees(np.arctan2(y_vals, x_vals)) % 360
            return angles.apply(lambda a: angle_to_octant(a) if np.isfinite(a) else np.nan).astype("Int64")
        except Exception:
            continue

    return pd.Series([np.nan] * len(trials_df), dtype="Float64")

In [51]:
def load_tc_data(nwb_path, rate_hz=None):
    """
    Load ThresholdCrossings data array and time axis.
    Returns (data: ndarray T×C, times: ndarray T, rate: float).
    """
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()
        for obj in nwb.all_children():
            if safe_attr(obj, "name") != "ThresholdCrossings":
                continue
            data = safe_attr(obj, "data", None)
            if data is None:
                continue
            data = np.array(data, dtype=np.float32)      # T × C
            rate  = safe_attr(obj, "rate",          None)
            t0    = safe_attr(obj, "starting_time", 0.0) or 0.0
            ts    = safe_attr(obj, "timestamps",    None)

            if ts is not None:
                times = np.array(ts, dtype=np.float64)
            elif rate is not None and rate > 0:
                times = t0 + np.arange(data.shape[0]) / float(rate)
            else:
                # Fallback: assume 1 sample = 1 ms
                times = t0 + np.arange(data.shape[0]) * 1e-3

            return data, times, float(rate) if rate else 1000.0

    return None, None, None

In [52]:
def extract_trial_features(tc_data, times, trials_df, win_start=0.30, win_end=0.50):
    """
    Bin TC counts per trial within the decoding window.

    Parameters
    ----------
    tc_data   : (T, C) float32  — raw ThresholdCrossings array
    times     : (T,) float64    — absolute timestamps (s)
    trials_df : DataFrame       — must have start_time, stop_time columns
    win_start : float           — seconds after trial start (default 0.30 s)
    win_end   : float           — seconds after trial start (default 0.50 s)

    Returns
    -------
    X : (n_trials, C) float32   — spike count features
    valid_mask : bool array     — True where window was found in data
    """
    n_trials  = len(trials_df)
    n_chans   = tc_data.shape[1]
    X         = np.full((n_trials, n_chans), np.nan, dtype=np.float32)
    valid_mask = np.zeros(n_trials, dtype=bool)

    t_start_arr = trials_df["start_time"].values.astype(float)
    t_stop_arr  = trials_df["stop_time"].values.astype(float)
    t_min       = times[0]
    t_max       = times[-1]

    for i, (t0, t_stop) in enumerate(zip(t_start_arr, t_stop_arr)):
        w0 = t0 + win_start
        w1 = t0 + win_end

        # Fallback: use full trial if window exceeds trial duration
        trial_dur = t_stop - t0
        if trial_dur < (win_end - win_start):
            w0 = t0
            w1 = t_stop

        if w0 > t_max or w1 < t_min:
            continue

        idx = np.where((times >= w0) & (times < w1))[0]
        if len(idx) == 0:
            continue

        # Spike count = sum of absolute values (TC can be signed or unsigned)
        window_data = tc_data[idx, :]
        X[i, :]     = np.sum(np.abs(window_data), axis=0)
        valid_mask[i] = True

    return X, valid_mask

In [53]:
def poisson_nb_decoder(X_train, y_train, X_test, min_rate=0.25, n_classes=8):
    """
    Poisson Naïve Bayes decoder (Issar et al. 2020 §Methods).

    Parameters
    ----------
    X_train, X_test : (n, C) spike counts
    y_train         : (n,) integer class labels 0..n_classes-1
    min_rate        : channels with mean count below this are dropped
    n_classes       : number of target directions

    Returns
    -------
    y_pred : (n_test,) predicted labels
    n_units : number of channels kept
    """
    classes = np.arange(n_classes)
    EPS     = 1e-8

    # Compute mean spike count per class per channel
    lam = np.zeros((n_classes, X_train.shape[1]))
    for c in classes:
        mask = y_train == c
        if mask.sum() == 0:
            lam[c, :] = EPS
        else:
            lam[c, :] = np.maximum(X_train[mask].mean(axis=0), EPS)

    # Drop low-firing units (paper: < 0.25 spikes/s across all trials)
    mean_rate = X_train.mean(axis=0)
    keep      = mean_rate >= min_rate
    if keep.sum() == 0:
        keep = np.ones(X_train.shape[1], dtype=bool)

    lam_k     = lam[:, keep]
    X_test_k  = X_test[:, keep]

    # Poisson log-likelihood: sum_u [ n_u * log(λ_cu) - λ_cu ]
    n_test    = X_test_k.shape[0]
    log_probs = np.zeros((n_test, n_classes))
    for c in classes:
        log_probs[:, c] = (
            X_test_k * np.log(lam_k[c, :] + EPS) - lam_k[c, :]
        ).sum(axis=1)

    y_pred = np.argmax(log_probs, axis=1)
    return y_pred, int(keep.sum())

In [54]:
def cross_validate_decoder(X, y, n_folds=5, min_rate=0.25, n_classes=8):
    """
    5-fold stratified cross-validation of Poisson NB decoder.

    Returns
    -------
    acc    : float  — mean accuracy across folds
    std    : float  — std of per-fold accuracy
    n_units: int    — median units kept
    """
    n       = len(y)
    idx     = np.arange(n)
    np.random.shuffle(idx)
    folds   = np.array_split(idx, n_folds)

    fold_accs   = []
    fold_units  = []

    for k in range(n_folds):
        test_idx  = folds[k]
        train_idx = np.concatenate([folds[j] for j in range(n_folds) if j != k])

        X_train, y_train = X[train_idx], y[train_idx]
        X_test,  y_test  = X[test_idx],  y[test_idx]

        # Only keep folds with enough class coverage
        classes_present = np.unique(y_train)
        if len(classes_present) < n_classes:
            continue

        y_pred, n_units = poisson_nb_decoder(
            X_train, y_train, X_test,
            min_rate=min_rate, n_classes=n_classes
        )
        fold_accs.append((y_pred == y_test).mean())
        fold_units.append(n_units)

    if len(fold_accs) == 0:
        return np.nan, np.nan, 0

    return float(np.mean(fold_accs)), float(np.std(fold_accs)), int(np.median(fold_units))

## Inspect sample NWB — confirm stream structure before full run

Before decoding all 312 sessions we verify that the ThresholdCrossings stream
is accessible, the timestamps / rate are sensible, and the trial table contains
the expected timing and target columns.

In [55]:
print("Probing sample NWB:", SAMPLE_NWB.name)

with NWBHDF5IO(str(SAMPLE_NWB), mode="r", load_namespaces=True) as io:
    nwb = io.read()

    # ── ThresholdCrossings ──────────────────────────────────────────────────
    tc_obj = None
    for obj in nwb.all_children():
        if safe_attr(obj, "name") == "ThresholdCrossings":
            tc_obj = obj
            break

    assert tc_obj is not None, "ThresholdCrossings not found in sample NWB"

    tc_shape    = np.array(tc_obj.data).shape
    tc_rate     = safe_attr(tc_obj, "rate",          None)
    tc_t0       = safe_attr(tc_obj, "starting_time", 0.0)
    tc_ts       = safe_attr(tc_obj, "timestamps",    None)
    tc_unit     = safe_attr(tc_obj, "unit",          "unknown")
    tc_has_ts   = tc_ts is not None

    print("\n── ThresholdCrossings ──────────────────")
    print(f"  Shape       : {tc_shape}")
    print(f"  Rate (Hz)   : {tc_rate}")
    print(f"  Starting t  : {tc_t0}")
    print(f"  Has timestamps: {tc_has_ts}")
    print(f"  Unit        : {tc_unit}")
    if tc_has_ts:
        ts_arr = np.array(tc_ts)
        print(f"  Timestamps  : [{ts_arr[0]:.4f} … {ts_arr[-1]:.4f}] s")
        print(f"  Median dt   : {np.median(np.diff(ts_arr))*1000:.3f} ms → rate ≈ {1/np.median(np.diff(ts_arr)):.1f} Hz")

    # ── Trials ──────────────────────────────────────────────────────────────
    assert nwb.trials is not None, "No trials table in sample NWB"
    trials_sample = nwb.trials.to_dataframe()
    print("\n── Trials table ────────────────────────")
    print(f"  Shape  : {trials_sample.shape}")
    print(f"  Columns: {list(trials_sample.columns)}")
    display(trials_sample.head(5))

Probing sample NWB: sub-Monkey-N_ses-20200127_ecephys.nwb

── ThresholdCrossings ──────────────────
  Shape       : (27649, 96)
  Rate (Hz)   : None
  Starting t  : None
  Has timestamps: True
  Unit        : volts
  Timestamps  : [8.4100 … 561.3700] s
  Median dt   : 20.000 ms → rate ≈ 50.0 Hz

── Trials table ────────────────────────
  Shape  : (375, 10)
  Columns: ['start_time', 'stop_time', 'trial_number', 'trial_count', 'run_id', 'index_target_position', 'mrs_target_position', 'target_style', 'trial_timeout', 'timeseries']


,start_time,stop_time,trial_number,trial_count,run_id,index_target_position,mrs_target_position,target_style,trial_timeout,timeseries
id,,,,,,,,,,
0,8.41,9.67,6,64,3,0.3,0.3,CO,30000,"[(0, 63, index_position pynwb.base.TimeSeries ..."
1,9.69,11.05,7,69,3,0.5,0.5,CO,30000,"[(64, 68, index_position pynwb.base.TimeSeries..."
2,11.07,12.95,8,95,3,0.7,0.5,CO,30000,"[(133, 94, index_position pynwb.base.TimeSerie..."
3,12.97,13.77,9,41,3,0.5,0.5,CO,30000,"[(228, 40, index_position pynwb.base.TimeSerie..."
4,13.79,17.01,10,162,3,0.2,0.8,CO,30000,"[(269, 161, index_position pynwb.base.TimeSeri..."


In [56]:
print("Probing sample NWB:", SAMPLE_NWB.name)

with NWBHDF5IO(str(SAMPLE_NWB), mode="r", load_namespaces=True) as io:
    nwb = io.read()

    tc_obj = None
    for obj in nwb.all_children():
        if safe_attr(obj, "name") == "ThresholdCrossings":
            tc_obj = obj
            break

    assert tc_obj is not None, "ThresholdCrossings not found in sample NWB"

    # Eagerly load everything we need BEFORE the file closes
    tc_shape = np.array(tc_obj.data).shape
    tc_rate  = safe_attr(tc_obj, "rate", None)
    tc_t0    = safe_attr(tc_obj, "starting_time", 0.0)
    tc_unit  = safe_attr(tc_obj, "unit", "unknown")

    # ── CRITICAL: materialise timestamps into a numpy array while file is open ──
    _ts_raw = safe_attr(tc_obj, "timestamps", None)
    if _ts_raw is not None:
        try:
            tc_ts_arr = np.array(_ts_raw[:], dtype=np.float64)
        except Exception:
            # pynwb may have already converted it; try direct cast
            tc_ts_arr = np.asarray(list(_ts_raw), dtype=np.float64)
        tc_has_ts = True
    else:
        tc_ts_arr = None
        tc_has_ts = False

    # ── Trials ──
    trials_df_sample = nwb.trials.to_dataframe() if nwb.trials is not None else pd.DataFrame()

print("\n── ThresholdCrossings ──────────────────")
print(f"  Shape       : {tc_shape}")
print(f"  Rate (Hz)   : {tc_rate}")
print(f"  Starting t0 : {tc_t0}")
print(f"  Unit        : {tc_unit}")
print(f"  Has timestamps: {tc_has_ts}")
if tc_has_ts:
    print(f"  ts_arr[0:3] : {tc_ts_arr[:3]}")
    print(f"  ts_arr[-3:] : {tc_ts_arr[-3:]}")
    print(f"  ts dtype    : {tc_ts_arr.dtype}")
print(f"\nTrials sample shape: {trials_df_sample.shape}")
display(trials_df_sample.head(3))

Probing sample NWB: sub-Monkey-N_ses-20200127_ecephys.nwb

── ThresholdCrossings ──────────────────
  Shape       : (27649, 96)
  Rate (Hz)   : None
  Starting t0 : None
  Unit        : volts
  Has timestamps: True
  ts_arr[0:3] : [8.41 8.43 8.45]
  ts_arr[-3:] : [561.33 561.35 561.37]
  ts dtype    : float64

Trials sample shape: (375, 10)


,start_time,stop_time,trial_number,trial_count,run_id,index_target_position,mrs_target_position,target_style,trial_timeout,timeseries
id,,,,,,,,,,
0,8.41,9.67,6,64,3,0.3,0.3,CO,30000,"[(0, 63, index_position pynwb.base.TimeSeries ..."
1,9.69,11.05,7,69,3,0.5,0.5,CO,30000,"[(64, 68, index_position pynwb.base.TimeSeries..."
2,11.07,12.95,8,95,3,0.7,0.5,CO,30000,"[(133, 94, index_position pynwb.base.TimeSerie..."


In [57]:
# ── Infer effective sampling rate ────────────────────────────────────────────
if tc_has_ts and tc_ts_arr is not None and len(tc_ts_arr) > 1:
    # tc_ts_arr is already a plain numpy array — no file I/O needed
    ts_sorted = np.sort(tc_ts_arr)
    diffs     = np.diff(ts_sorted)
    diffs     = diffs[diffs > 0]          # drop any zero-gap artefacts
    dt_med    = float(np.median(diffs))
    RATE_HZ   = 1.0 / dt_med
elif tc_rate is not None and float(tc_rate) > 0:
    RATE_HZ   = float(tc_rate)
else:
    RATE_HZ   = 1000.0                    # safe fallback: 1 kHz

print(f"Effective sampling rate used for decoding: {RATE_HZ:.2f} Hz")

# ── Compute window width in samples ─────────────────────────────────────────
WIN_START = 0.30   # s after trial start
WIN_END   = 0.50   # s after trial start
WIN_SAMPS = int((WIN_END - WIN_START) * RATE_HZ)
print(f"Decoding window: [{WIN_START*1000:.0f}, {WIN_END*1000:.0f}] ms  → {WIN_SAMPS} samples")

Effective sampling rate used for decoding: 50.00 Hz
Decoding window: [300, 500] ms  → 10 samples


In [58]:
# ── Test target-label extraction on sample ────────────────────────────────
labels_sample = extract_target_labels(trials_sample)
print("Label sample (first 20):", labels_sample.head(20).tolist())
print("Unique labels:", sorted(labels_sample.dropna().unique().tolist()))
print("Label counts:\n", labels_sample.value_counts().sort_index())

TypeError: cannot safely cast non-equivalent object to int64

In [ ]:
# ── Quick decode test on sample NWB ─────────────────────────────────────────
tc_data, times, rate_detected = load_tc_data(SAMPLE_NWB)
print(f"TC data shape  : {tc_data.shape}")
print(f"Times range    : [{times[0]:.4f}, {times[-1]:.4f}] s")
print(f"Rate detected  : {rate_detected:.2f} Hz")

# Extract features
X_sample, valid_mask = extract_trial_features(tc_data, times, trials_sample,
                                               win_start=WIN_START, win_end=WIN_END)
y_sample_raw = extract_target_labels(trials_sample).values

# Keep only valid trials with a label
keep = valid_mask & np.array([v is not None and not (isinstance(v, float) and np.isnan(v))
                                for v in y_sample_raw])
X_valid = X_sample[keep].astype(np.float32)
y_valid = y_sample_raw[keep].astype(int)

print(f"\nValid trials for decoding : {keep.sum()} / {len(trials_sample)}")
print(f"X_valid shape             : {X_valid.shape}")
print(f"y_valid unique classes    : {sorted(np.unique(y_valid).tolist())}")

acc_sample, std_sample, units_sample = cross_validate_decoder(
    X_valid, y_valid, n_folds=5, n_classes=len(np.unique(y_valid))
)
print(f"\nSample session baseline accuracy : {acc_sample*100:.2f}% ± {std_sample*100:.2f}%")
print(f"Chance level                     : {CHANCE*100:.1f}%")
print(f"Units kept                       : {units_sample}")

## Figure 1 — Sample session accuracy report (paper Fig. 4A/B, γ = 0 baseline)

This figure reproduces the **γ = 0 cross on the dual-axis plot** from Fig. 4A/B of
Issar et al. (2020).  It shows a single session's decoding accuracy at the threshold-
crossings baseline, the chance level, and the number of units contributing to decoding.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

n_classes_sample = len(np.unique(y_valid))
chance_here      = 1.0 / n_classes_sample

ax.bar(["Baseline\n(γ = 0)", "Chance"],
       [acc_sample * 100, chance_here * 100],
       color=[C_MAROON, C_GRAY],
       edgecolor="black", linewidth=0.9, width=0.45)

ax.errorbar(["Baseline\n(γ = 0)"], [acc_sample * 100],
            yerr=[std_sample * 100],
            fmt="none", color="black", capsize=6, linewidth=1.2)

ax.axhline(chance_here * 100, color=C_GRAY, linestyle="--", linewidth=1.2, label="Chance")

ax.set_ylabel("Decoding accuracy (%)")
ax.set_title(
    f"Sample session baseline decoder\n"
    f"sub-Monkey-N_ses-20200127  |  "
    f"γ = 0 (all threshold crossings)  |  {units_sample} units  |  {keep.sum()} trials",
    pad=10
)
paper_axes(ax)
ax.set_ylim(0, max(acc_sample * 100 * 1.5, chance_here * 100 * 2.5, 30))

from matplotlib.patches import Patch
leg_items = [
    Patch(facecolor=C_MAROON, edgecolor="black", label=f"TC baseline: {acc_sample*100:.1f}%"),
    Patch(facecolor=C_GRAY,   edgecolor="black", label=f"Chance: {chance_here*100:.1f}%"),
]
ax.legend(handles=leg_items, loc="upper right", frameon=True, fancybox=False)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig01_sample_session_baseline_accuracy.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig01_sample_session_baseline_accuracy.pdf", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved fig01")

## Full decoder run — all 312 sessions

We now decode all sessions using the Poisson NB decoder at γ = 0.
Each session is processed independently:
1. Load ThresholdCrossings stream
2. Extract trial-aligned spike counts in [300, 500] ms window
3. Discretise target labels into 8 octants
4. 5-fold cross-validate Poisson NB decoder
5. Record per-session accuracy

In [ ]:
def decode_one_session(nwb_path, trials_df, win_start=0.30, win_end=0.50,
                        min_rate=0.25, n_folds=5):
    """Full pipeline for one session. Returns dict of results."""
    result = {
        "file_name"   : nwb_path.name,
        "n_trials_raw": len(trials_df),
        "n_valid"     : 0,
        "n_classes"   : 0,
        "n_units"     : 0,
        "acc_mean"    : np.nan,
        "acc_std"     : np.nan,
        "status"      : "ok",
    }

    # Load TC data
    tc_data, times, rate = load_tc_data(nwb_path)
    if tc_data is None:
        result["status"] = "no_tc_stream"
        return result

    # Extract features
    X, valid_mask = extract_trial_features(tc_data, times, trials_df,
                                           win_start=win_start, win_end=win_end)

    # Target labels
    y_raw = extract_target_labels(trials_df).values

    # Build valid mask
    label_valid = np.array([
        v is not None
        and not (isinstance(v, float) and np.isnan(v))
        and pd.notna(v)
        for v in y_raw
    ])
    keep = valid_mask & label_valid

    if keep.sum() < 20:
        result["status"] = f"insufficient_trials({keep.sum()})"
        return result

    X_v = X[keep].astype(np.float32)
    y_v = y_raw[keep].astype(int)

    n_classes = len(np.unique(y_v))
    if n_classes < 2:
        result["status"] = "single_class"
        return result

    acc, std, n_units = cross_validate_decoder(
        X_v, y_v, n_folds=n_folds,
        min_rate=min_rate, n_classes=n_classes
    )

    result.update({
        "n_valid"  : int(keep.sum()),
        "n_classes": int(n_classes),
        "n_units"  : int(n_units),
        "acc_mean" : float(acc),
        "acc_std"  : float(std),
        "chance"   : float(1.0 / n_classes),
    })
    return result

In [ ]:
# Group trial table by file_name for fast lookup
trial_groups = {fname: grp for fname, grp in trial_df.groupby("file_name")}

nwb_files = sorted(DATASET_DIR.glob("*_ecephys.nwb"))
print(f"NWB files found: {len(nwb_files)}")

all_results = []
for i, nwb_path in enumerate(nwb_files, start=1):
    fname = nwb_path.name
    if fname not in trial_groups:
        all_results.append({
            "file_name": fname,
            "status": "no_trial_data",
            "acc_mean": np.nan,
        })
        continue

    if i % 50 == 0 or i == 1 or i == len(nwb_files):
        print(f"  Decoding {i}/{len(nwb_files)}: {fname}")

    res = decode_one_session(
        nwb_path,
        trial_groups[fname].copy(),
        win_start=WIN_START,
        win_end=WIN_END,
        min_rate=0.25,
        n_folds=5
    )
    all_results.append(res)

results_df = pd.DataFrame(all_results)

# Merge session-level metadata
merge_cols = [c for c in ["file_name", "session_index", "session_date",
                           "days_since_first_session", "n_trials", "n_units",
                           "file_size_mb", "target_style_inferred"]
              if c in session_df.columns]
results_df = results_df.merge(
    session_df[merge_cols].drop_duplicates("file_name"),
    on="file_name", how="left"
)

results_df["session_date"] = pd.to_datetime(results_df["session_date"], errors="coerce")
results_df = results_df.sort_values("session_index").reset_index(drop=True)
results_df["normalised_acc"] = results_df["acc_mean"] / results_df["acc_mean"].median()

results_df.to_csv(OUT_TABLE_DIR / "session_baseline_accuracy.csv", index=False)
print(f"\nDecoding complete. Total sessions: {len(results_df)}")
print(results_df["status"].value_counts())
display(results_df.head(10))

In [ ]:
ok_df = results_df[results_df["acc_mean"].notna()].copy()

print("─── Baseline Decoder Summary ───────────────────────────────")
print(f"  Sessions decoded successfully : {len(ok_df)}")
print(f"  Mean accuracy                : {ok_df['acc_mean'].mean()*100:.2f}%")
print(f"  Median accuracy              : {ok_df['acc_mean'].median()*100:.2f}%")
print(f"  Std accuracy                 : {ok_df['acc_mean'].std()*100:.2f}%")
print(f"  Chance level (1/8)           : {CHANCE*100:.1f}%")
print(f"  Sessions above chance        : {(ok_df['acc_mean'] > CHANCE).sum()} / {len(ok_df)}")
print(f"  Mean units used              : {ok_df['n_units'].mean():.1f}")

## Figure 2 — Session-wise accuracy and trial count (paper Fig. 4A/B dual-axis)

This dual-axis line plot shows decoding accuracy (left axis, maroon) and
trial count (right axis, red dashed) as a function of session index.
It is the longitudinal equivalent of the dual-axis plot in Fig. 4A/B of the paper,
adapted to show the γ = 0 baseline across all sessions rather than a single session's
threshold sweep.

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))

ax2 = ax1.twinx()

# ── Left axis: decoding accuracy ────────────────────────────────────────────
ax1.plot(
    ok_df["session_index"],
    ok_df["acc_mean"] * 100,
    color=C_BLACK,
    linewidth=1.8,
    marker=MARKER_TC,
    markersize=4,
    markerfacecolor="white",
    markeredgewidth=0.8,
    label="Decoding accuracy (TC, γ=0)",
    zorder=3
)

ax1.axhline(CHANCE * 100, color=C_GRAY, linestyle="--", linewidth=1.3,
            label=f"Chance ({CHANCE*100:.1f}%)", zorder=2)

ax1.set_xlabel("Session index")
ax1.set_ylabel("Decoding accuracy (%)", color=C_BLACK)
ax1.tick_params(axis="y", colors=C_BLACK, which="both", direction="in")

# ── Right axis: trial count ──────────────────────────────────────────────────
if "n_trials" in ok_df.columns:
    trial_y = ok_df["n_trials"]
elif "n_valid" in ok_df.columns:
    trial_y = ok_df["n_valid"]
else:
    trial_y = pd.Series(np.nan, index=ok_df.index)

ax2.plot(
    ok_df["session_index"],
    trial_y,
    color=C_MAROON,
    linewidth=1.4,
    linestyle="--",
    marker="o",
    markersize=3,
    markerfacecolor="white",
    markeredgewidth=0.7,
    label="Trials per session",
    zorder=2
)

ax2.set_ylabel("Trials per session", color=C_MAROON)
ax2.tick_params(axis="y", colors=C_MAROON, which="both", direction="in")

ax1.set_title(
    "Dual-axis: Baseline decoding accuracy (γ = 0) and trial count per session",
    pad=10
)

paper_axes(ax1, top=True, right=False)
ax1.grid(True, which="major", linestyle=":", linewidth=0.8, alpha=0.85)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", frameon=True, fancybox=False)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig02_session_accuracy_dual_axis.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig02_session_accuracy_dual_axis.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Figure 3 — Decoding accuracy vs days since first session (paper Fig. 5C)

This is the exact analogue of **Fig. 5C** in Issar et al. (2020):
> *"Decoding accuracy with threshold crossings (i.e., γ = 0) decreased as the array
> aged (Spearman's correlation, monkey Pe ρ = −0.86, P < 0.0001)."*

We compute Spearman's ρ and overlay the OLS regression line.
Early sessions (≤ 180 days) use filled circles; late sessions use open circles —
matching the paper's Fig. 5D convention.

In [ ]:
plot_df = ok_df[ok_df["days_since_first_session"].notna()].copy()

days  = plot_df["days_since_first_session"].values.astype(float)
accs  = plot_df["acc_mean"].values.astype(float) * 100

rho, pval = stats.spearmanr(days, accs)
slope, intercept, _, _, _ = stats.linregress(days, accs)
x_line = np.array([days.min(), days.max()])
y_line = slope * x_line + intercept

early_mask = days <= 180
late_mask  = ~early_mask

fig, ax = plt.subplots(figsize=(11, 7))

# ── Late sessions ────────────────────────────────────────────────────────────
ax.scatter(
    days[late_mask], accs[late_mask],
    s=34, color=C_MAROON, marker=MARKER_TC,
    edgecolor="black", linewidth=0.5, zorder=3,
    label="Late sessions (> 180 days)"
)

# ── Early sessions ───────────────────────────────────────────────────────────
ax.scatter(
    days[early_mask], accs[early_mask],
    s=34, color=C_MAROON, marker=MARKER_TC,
    edgecolor="black", linewidth=0.5, zorder=3, facecolors="white",
    label="Early sessions (≤ 180 days)"
)

# ── Regression line ───────────────────────────────────────────────────────────
p_str = f"{pval:.2e}" if pval >= 1e-10 else f"< 1e-10"
ax.plot(x_line, y_line, color=C_BLACK, linewidth=2.0,
        label=f"ρ = {rho:.2f}, p = {p_str}")

ax.axhline(CHANCE * 100, color=C_GRAY, linestyle="--", linewidth=1.2,
           label=f"Chance ({CHANCE*100:.1f}%)")

ax.set_xlabel("Days since first session")
ax.set_ylabel("Decoding accuracy, TC (%, γ = 0)")
ax.set_title(
    "A   ThresholdCrossings: decoding accuracy vs days since first session\n"
    "(Baseline decoder, γ = 0, Poisson Naïve Bayes, 5-fold CV)",
    pad=10, loc="left"
)
paper_axes(ax)
leg = ax.legend(loc="upper right", frameon=True, fancybox=False)
leg.get_frame().set_linewidth(0.9)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig03_accuracy_vs_days.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig03_accuracy_vs_days.pdf", dpi=300, bbox_inches="tight")
plt.show()
print(f"Spearman ρ = {rho:.3f},  p = {pval:.3e}")

## Figure 4 — Early vs late session accuracy distributions (paper Fig. 5D)

**Fig. 5D** in the paper shows Δ decoding accuracy (NAS − TC) per session, split
into early (≤ 50 days) and late (> 50 days) groups.  Here we show the **raw TC
baseline accuracy** split by the same early/late boundary, which is the reference
against which NAS improvements will be measured in Notebook 05.

In [ ]:
EARLY_CUTOFF = 180    # days — using our dataset's scale (paper: 50 days)

early_df = ok_df[ok_df["days_since_first_session"] <= EARLY_CUTOFF]
late_df  = ok_df[ok_df["days_since_first_session"] >  EARLY_CUTOFF]

early_accs = early_df["acc_mean"].dropna().values * 100
late_accs  = late_df["acc_mean"].dropna().values  * 100

stat, pval_wsr = stats.ranksums(early_accs, late_accs)

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=False)

for ax, vals, label, color in [
    (axes[0], early_accs, f"Early (≤ {EARLY_CUTOFF} d)\nn = {len(early_accs)}", C_EARLY),
    (axes[1], late_accs,  f"Late (> {EARLY_CUTOFF} d)\nn = {len(late_accs)}",  C_LATE),
]:
    ax.hist(vals, bins=25, color=color, edgecolor="black", linewidth=0.6, alpha=0.9)
    med = np.median(vals)
    ax.axvline(med, color=C_MAROON, linestyle="--", linewidth=1.5,
               label=f"Median = {med:.1f}%")
    ax.axvline(CHANCE * 100, color=C_GRAY, linestyle=":", linewidth=1.2,
               label=f"Chance = {CHANCE*100:.1f}%")
    ax.set_xlabel("Decoding accuracy, TC (%)")
    ax.set_ylabel("Number of sessions")
    ax.set_title(label, pad=8)
    paper_axes(ax)
    leg = ax.legend(loc="upper right", frameon=True, fancybox=False, fontsize=10)
    leg.get_frame().set_linewidth(0.9)

fig.suptitle(
    f"D   Baseline accuracy distribution: early vs late sessions\n"
    f"(Wilcoxon rank-sum p = {pval_wsr:.3e})",
    y=1.02, fontsize=13
)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig04_early_vs_late_distributions.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig04_early_vs_late_distributions.pdf", dpi=300, bbox_inches="tight")
plt.show()
print(f"Wilcoxon rank-sum: stat = {stat:.3f},  p = {pval_wsr:.3e}")

## Figure 5 — Session distribution histograms, 4-panel (paper Fig. 4C–F)

These four histograms mirror **Fig. 4C–F** exactly, which the paper uses to show the
distribution of γ-thresholds and Δ decoding accuracy across sessions.  For the baseline
notebook, the four panels show the distributions of: (C) decoding accuracy, (D) units
used, (E) valid trials, (F) days since first session.

In [ ]:
fig = plt.figure(figsize=(13, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.32)

panel_defs = [
    ("C", "acc_mean", "Decoding accuracy (γ = 0)",
     lambda v: v * 100, "Accuracy (%)", C_MAROON),
    ("D", "n_units",  "Active units per session",
     lambda v: v,      "Number of units", C_BLUE),
    ("E", "n_valid",  "Valid trials per session",
     lambda v: v,      "Valid trials", C_EARLY),
    ("F", "days_since_first_session", "Days since first session",
     lambda v: v,      "Days", C_GRAY),
]

for idx, (panel_label, col, title, transform, xlabel, color) in enumerate(panel_defs):
    ax = fig.add_subplot(gs[idx // 2, idx % 2])

    vals = ok_df[col].dropna().apply(transform).values
    ax.hist(vals, bins=30, color=color, edgecolor="black", linewidth=0.6, alpha=0.9)

    med = np.median(vals)
    ax.axvline(med, color=C_MAROON, linestyle="--", linewidth=1.5,
               label=f"Median = {med:.1f}")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of sessions")
    ax.set_title(f"{panel_label}   {title}", pad=8, loc="left")
    paper_axes(ax)
    ax.legend(loc="upper right", frameon=True, fancybox=False, fontsize=10)

plt.savefig(OUT_FIG_DIR / "fig05_session_distribution_4panel.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig05_session_distribution_4panel.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Figure 6 — Mean ± 1 SE normalised accuracy: early vs late (paper Fig. 5E)

**Fig. 5E** in the paper shows mean normalised decoding accuracy ± 1 SE as a function of
γ-threshold, separately for early and late sessions.  The key finding is that late sessions
benefit *more* from NAS than early sessions.

Here we show the **session-rank** version of this plot (γ is constant at 0),
which establishes the baseline trajectory that Notebook 05 will compare against.

In [ ]:
ok_sorted = ok_df.sort_values("session_index").reset_index(drop=True)
med_acc   = ok_sorted["acc_mean"].median()
ok_sorted["norm_acc"] = ok_sorted["acc_mean"] / (med_acc if med_acc > 0 else 1.0)

SPLIT_DAY = 180
early_mask_s = ok_sorted["days_since_first_session"] <= SPLIT_DAY
late_mask_s  = ~early_mask_s

early_sorted = ok_sorted[early_mask_s].reset_index(drop=True)
late_sorted  = ok_sorted[late_mask_s].reset_index(drop=True)

# Rolling mean ± SE for late (enough points)
def rolling_mean_se(arr, window=15):
    s = pd.Series(arr)
    m = s.rolling(window, center=True, min_periods=3).mean()
    e = s.rolling(window, center=True, min_periods=3).sem()
    return m.values, e.values

late_m, late_e = rolling_mean_se(late_sorted["norm_acc"].values)

fig, ax = plt.subplots(figsize=(13, 6))

# ── Late sessions: shaded mean ± 1 SE ────────────────────────────────────────
ranks_late = np.arange(len(late_sorted))
ax.plot(ranks_late, late_m, color=C_LATE, linewidth=2.2, marker="o", ms=4,
        markerfacecolor="white", markeredgewidth=0.8,
        label=f"Late sessions (> {SPLIT_DAY} d)")
ax.fill_between(ranks_late,
                late_m - late_e, late_m + late_e,
                color=C_LATE, alpha=0.20)

# ── Early sessions: individual points (too few for rolling) ──────────────────
if len(early_sorted) > 0:
    ranks_early = np.arange(len(early_sorted))
    early_m, early_e = rolling_mean_se(early_sorted["norm_acc"].values, window=5)
    ax.plot(ranks_early, early_m, color=C_EARLY, linewidth=2.2, marker="o", ms=4,
            markerfacecolor="white", markeredgewidth=0.8,
            label=f"Early sessions (≤ {SPLIT_DAY} d)")
    ax.fill_between(ranks_early,
                    early_m - early_e, early_m + early_e,
                    color=C_EARLY, alpha=0.20)

ax.axhline(1.0, color=C_GRAY, linestyle="--", linewidth=1.2,
           label="Overall median (reference)")

ax.set_xlabel("Session rank within epoch")
ax.set_ylabel("Normalised decoding accuracy\n(relative to overall median)")
ax.set_title(
    "E   Mean normalised baseline accuracy (γ = 0): early vs late sessions\n"
    "(shading = ± 1 SE)",
    pad=10, loc="left"
)
paper_axes(ax)
leg = ax.legend(loc="upper right", frameon=True, fancybox=False)
leg.get_frame().set_linewidth(0.9)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig06_normalised_accuracy_early_late.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig06_normalised_accuracy_early_late.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Figure 7 — Decoding accuracy vs recording quality proxies (paper Fig. 5A/B)

**Fig. 5A/B** shows % low-Pspike and % high-Pspike waveforms vs array age.
Since NAS is not yet trained, we use two available quality proxies:
- **File size (MB)** — correlates with data volume (more waveforms → larger file)
- **T-samples** — encodes recording duration

Both are plotted against decoding accuracy with Spearman ρ annotation,
matching the scatter-with-regression style of Fig. 5A/B.

In [ ]:
proxy_cols = [c for c in ["file_size_mb", "T_samples_tc"] if c in ok_df.columns]

# Merge T-samples from stream manifest if available
if "T_samples_tc" not in ok_df.columns and "T_samples" in stream_df.columns:
    tc_T = (
        stream_df[stream_df["name"] == "ThresholdCrossings"]
        [["file_name", "T_samples"]]
        .rename(columns={"T_samples": "T_samples_tc"})
    )
    ok_df = ok_df.merge(tc_T, on="file_name", how="left")
    proxy_cols = [c for c in ["file_size_mb", "T_samples_tc"] if c in ok_df.columns]

if len(proxy_cols) == 0:
    proxy_cols = ["days_since_first_session"]

fig, axes = plt.subplots(len(proxy_cols), 1,
                          figsize=(11, 5 * len(proxy_cols)), squeeze=False)

proxy_labels = {
    "file_size_mb"         : "File size (MB)",
    "T_samples_tc"         : "T-samples (ThresholdCrossings)",
    "days_since_first_session": "Days since first session",
}

for ax_row, col in zip(axes, proxy_cols):
    ax = ax_row[0]
    sub = ok_df[[col, "acc_mean"]].dropna()
    x  = sub[col].values.astype(float)
    y  = sub["acc_mean"].values * 100

    rho_p, pval_p = stats.spearmanr(x, y)
    slope_p, intercept_p, *_ = stats.linregress(x, y)
    x_line = np.array([x.min(), x.max()])
    y_line = slope_p * x_line + intercept_p

    ax.scatter(x, y, s=28, color=C_MAROON, marker=MARKER_TC,
               edgecolor="black", linewidth=0.5, zorder=3,
               label="ThresholdCrossings")
    ax.plot(x_line, y_line, color=C_BLACK, linewidth=2.0,
            label=f"ρ = {rho_p:.2f}, p = {pval_p:.2e}")
    ax.axhline(CHANCE * 100, color=C_GRAY, linestyle="--", linewidth=1.2,
               label=f"Chance ({CHANCE*100:.1f}%)")

    panel_letter = chr(65 + proxy_cols.index(col))
    ax.set_xlabel(proxy_labels.get(col, col))
    ax.set_ylabel("Decoding accuracy, TC (%)")
    ax.set_title(f"{panel_letter}   Accuracy